# MorphoFusion-MLDL

Código fuente del sistema para la clasificación del nivel de rendimiento deportivo mediante datos biomédicos morfológicos, Machine Learning, Deep Learning e interpretabilidad con SHAP.


In [ ]:
# Instalar dependencias necesarias en Google Colab
!pip install -q xgboost shap openpyxl


In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

from google.colab import files

In [ ]:
uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]
nombre_archivo

In [ ]:
df_original = pd.read_excel(nombre_archivo, sheet_name="Hoja1")

print("Filas:", df_original.shape[0])
print("Columnas:", df_original.shape[1])

df_original.head()

In [ ]:
def limpiar_nombre_columna(nombre):
    texto = str(nombre).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join([c for c in texto if not unicodedata.combining(c)])
    texto = texto.replace("%", "porcentaje")
    texto = re.sub(r"[^a-zA-Z0-9]+", "_", texto)
    texto = re.sub(r"_+", "_", texto).strip("_")
    return texto

df = df_original.copy()
df.columns = [limpiar_nombre_columna(col) for col in df.columns]

df.head()

In [ ]:
faltantes = (
    df.isnull()
    .sum()
    .reset_index()
    .rename(columns={"index": "Variable", 0: "Valores faltantes"})
)

faltantes["Porcentaje faltante"] = (faltantes["Valores faltantes"] / len(df) * 100).round(2)
faltantes = faltantes.sort_values("Valores faltantes", ascending=False)

print("Duplicados:", df.duplicated().sum())
faltantes.head(20)

In [ ]:
df["nivel"].value_counts(dropna=False)

In [ ]:
df["nivel_limpio"] = (
    df["nivel"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["nivel_limpio"] = df["nivel_limpio"].replace({
    "PRICIPIANTE": "PRINCIPIANTE"
})

mapa_rendimiento = {
    "PRINCIPIANTE": "En desarrollo",
    "INTERMEDIO": "En desarrollo",
    "AVANZADO": "Alto rendimiento",
    "RENDIMIENTO": "Alto rendimiento",
    "EXPERIMENTADO": "Alto rendimiento",
    "ALTO RENDIMIENTO": "Alto rendimiento",
    "TITULAR": "Alto rendimiento"
}

df["rendimiento_clase"] = df["nivel_limpio"].map(mapa_rendimiento)

df["rendimiento_binario"] = df["rendimiento_clase"].map({
    "En desarrollo": 0,
    "Alto rendimiento": 1
})

df["rendimiento_clase"].value_counts()

In [ ]:
columnas_excluir = [
    "n",
    "nombre_y_apellidos",
    "universidad",
    "deporte",
    "deportes_aplicar",
    "fecha_eval",
    "fecha_nac",
    "nivel",
    "nivel_limpio",
    "rendimiento_clase",
    "rendimiento_binario"
]

# También excluimos columnas textuales de interpretación, evaluación o resultado
columnas_textuales_excluir = [
    col for col in df.columns
    if any(palabra in col for palabra in [
        "interpretacion",
        "evaluacion",
        "resultado",
        "clasificacion",
        "tipo_de_programa",
        "deporte_adecuado"
    ])
]

columnas_excluir_final = list(set(columnas_excluir + columnas_textuales_excluir))

X_candidatas = df.drop(columns=columnas_excluir_final, errors="ignore")

print("Variables candidatas después de excluir identificadores y textos:", X_candidatas.shape[1])
X_candidatas.head()

In [ ]:
def asignar_dimension(variable):
    v = variable.lower()

    antropometria = [
        "peso", "talla", "envergadura", "estatura", "biacro", "bilio",
        "humer", "femor", "tibial", "radial", "acromial", "long",
        "perimetro", "brflex", "brrel", "antebr", "torax", "cintura",
        "cadera", "muslo", "pant", "pliegue", "trc", "ssc", "ssp",
        "abd", "mmed", "cbz"
    ]

    composicion = [
        "masa", "muscular", "adiposa", "osea", "grasa", "lipid",
        "musc", "adip", "piel", "area", "fraccion"
    ]

    proporcionalidad = [
        "imc", "indice", "rohrer", "cormico", "iecto", "irmi",
        "lrmi", "lrms", "lres", "braquial", "crural", "pignet",
        "relativa", "hwr", "endo", "meso", "ecto"
    ]

    maduracion = [
        "edad", "maduracion", "madurez", "osea", "morfologica",
        "pico_veloc", "crecimiento", "talla_diana",
        "altura_adulta", "proyeccion"
    ]

    if any(p in v for p in maduracion):
        return "Maduración biológica y crecimiento"
    elif any(p in v for p in composicion):
        return "Composición corporal"
    elif any(p in v for p in proporcionalidad):
        return "Proporcionalidad corporal"
    elif any(p in v for p in antropometria):
        return "Antropometría básica y segmentaria"
    else:
        return "Revisar manualmente"

In [ ]:
tabla_variables = pd.DataFrame({
    "Variable": X_candidatas.columns
})

tabla_variables["Dimensión morfológica"] = tabla_variables["Variable"].apply(asignar_dimension)

tabla_variables["Usar en el modelo"] = np.where(
    tabla_variables["Dimensión morfológica"] == "Revisar manualmente",
    "Revisar",
    "Sí"
)

tabla_variables

In [ ]:
tabla_variables["Dimensión morfológica"].value_counts()

In [ ]:
variables_usadas = tabla_variables.loc[
    tabla_variables["Usar en el modelo"] == "Sí",
    "Variable"
].tolist()

base_modelado = df[variables_usadas + ["rendimiento_clase", "rendimiento_binario"]].copy()

print("Filas:", base_modelado.shape[0])
print("Variables predictoras:", len(variables_usadas))
print("Columnas totales:", base_modelado.shape[1])

base_modelado.head()

In [ ]:
tabla_variables.to_excel("tabla_variables_morfologicas_usadas.xlsx", index=False)
base_modelado.to_csv("base_modelado_morphofusion_desde_cero.csv", index=False)

files.download("tabla_variables_morfologicas_usadas.xlsx")
files.download("base_modelado_morphofusion_desde_cero.csv")

In [ ]:
variables_revisar = tabla_variables[
    tabla_variables["Dimensión morfológica"] == "Revisar manualmente"
]

variables_revisar

In [ ]:
dimensiones_validas = [
    "Antropometría básica y segmentaria",
    "Composición corporal",
    "Proporcionalidad corporal",
    "Maduración biológica y crecimiento"
]

tabla_variables["Usar en el modelo"] = np.where(
    tabla_variables["Dimensión morfológica"].isin(dimensiones_validas),
    "Sí",
    "No"
)

tabla_variables["Usar en el modelo"].value_counts()

In [ ]:
resumen_dimensiones = (
    tabla_variables[tabla_variables["Usar en el modelo"] == "Sí"]
    ["Dimensión morfológica"]
    .value_counts()
    .reset_index()
)

resumen_dimensiones.columns = ["Dimensión morfológica", "Número de variables"]
resumen_dimensiones

In [ ]:
variables_finales = tabla_variables.loc[
    tabla_variables["Usar en el modelo"] == "Sí",
    "Variable"
].tolist()

base_modelado = df[variables_finales + ["rendimiento_clase", "rendimiento_binario"]].copy()

print("Filas:", base_modelado.shape[0])
print("Variables predictoras finales:", len(variables_finales))
print("Columnas totales:", base_modelado.shape[1])

base_modelado.head()

In [ ]:
palabras_prohibidas = [
    "nombre", "apellido", "universidad", "deporte", "fecha",
    "nivel", "rendimiento", "interpretacion", "evaluacion",
    "resultado", "clasificacion"
]

variables_sospechosas = [
    col for col in base_modelado.columns
    if any(palabra in col.lower() for palabra in palabras_prohibidas)
]

variables_sospechosas

In [ ]:
tabla_variables.to_excel("tabla_variables_morfologicas_finales.xlsx", index=False)
base_modelado.to_csv("base_modelado_morphofusion_final.csv", index=False)

files.download("tabla_variables_morfologicas_finales.xlsx")
files.download("base_modelado_morphofusion_final.csv")

In [ ]:
palabras_prohibidas = [
    "nombre", "apellido", "universidad", "deporte", "fecha",
    "nivel", "interpretacion", "evaluacion", "resultado",
    "clasificacion"
]

variables_sospechosas = [
    col for col in base_modelado.columns
    if any(palabra in col.lower() for palabra in palabras_prohibidas)
]

variables_sospechosas

In [ ]:
X = base_modelado.drop(
    columns=["rendimiento_clase", "rendimiento_binario"],
    errors="ignore"
)

y = base_modelado["rendimiento_binario"]

print("X:", X.shape)
print("y:", y.shape)

y.value_counts()

In [ ]:
tipos_variables = pd.DataFrame({
    "Variable": X.columns,
    "Tipo de dato": X.dtypes.astype(str)
})

tipos_variables["Tipo general"] = np.where(
    tipos_variables["Tipo de dato"].isin(["object", "string", "category"]),
    "Categórica",
    "Numérica"
)

tipos_variables["Tipo general"].value_counts()

In [ ]:
variables_numericas = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
variables_categoricas = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Variables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))

variables_categoricas

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("\nDistribución en entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución en prueba:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocesamiento = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline(steps=[
                ("imputacion", SimpleImputer(strategy="median")),
                ("escalamiento", StandardScaler())
            ]),
            variables_numericas
        ),
        (
            "categoricas",
            Pipeline(steps=[
                ("imputacion", SimpleImputer(strategy="most_frequent")),
                ("codificacion", OneHotEncoder(handle_unknown="ignore"))
            ]),
            variables_categoricas
        )
    ],
    remainder="drop"
)

preprocesamiento

In [ ]:
variables_categoricas

In [ ]:
X[variables_categoricas].head(10)

In [ ]:
for col in variables_categoricas:
    print("\nVariable:", col)
    print(X[col].value_counts(dropna=False))

In [ ]:
# Eliminar variable categórica derivada
X = X.drop(columns=["percentil_talla"], errors="ignore")

# Actualizar listas de variables
variables_numericas = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
variables_categoricas = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Variables finales después de excluir percentil_talla:")
print("Variables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))
print("Total predictoras:", X.shape[1])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocesamiento = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline(steps=[
                ("imputacion", SimpleImputer(strategy="median")),
                ("escalamiento", StandardScaler())
            ]),
            variables_numericas
        )
    ],
    remainder="drop"
)

preprocesamiento

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

resumen_dimensiones = (
    tabla_variables[
        (tabla_variables["Usar en el modelo"] == "Sí") &
        (tabla_variables["Variable"] != "percentil_talla")
    ]
    ["Dimensión morfológica"]
    .value_counts()
    .reset_index()
)

resumen_dimensiones.columns = ["Dimensión morfológica", "Número de variables"]

orden_dimensiones = [
    "Antropometría básica y segmentaria",
    "Composición corporal",
    "Proporcionalidad corporal",
    "Maduración biológica y crecimiento"
]

resumen_dimensiones["Dimensión morfológica"] = pd.Categorical(
    resumen_dimensiones["Dimensión morfológica"],
    categories=orden_dimensiones,
    ordered=True
)

resumen_dimensiones = resumen_dimensiones.sort_values("Dimensión morfológica")

plt.figure(figsize=(9, 5), dpi=150)

sns.barplot(
    data=resumen_dimensiones,
    x="Número de variables",
    y="Dimensión morfológica",
    color="#4B6F8C"
)

plt.title("Distribución de variables predictoras por dimensión morfológica", fontsize=13, weight="bold")
plt.xlabel("Número de variables")
plt.ylabel("")

for index, value in enumerate(resumen_dimensiones["Número de variables"]):
    plt.text(value + 0.5, index, str(value), va="center", fontsize=10)

plt.tight_layout()
plt.savefig("figura_variables_por_dimension_morfologica.png", dpi=300, bbox_inches="tight")
plt.show()

resumen_dimensiones

In [ ]:
# Actualizar variables finales eliminando percentil_talla
variables_finales = tabla_variables.loc[
    tabla_variables["Usar en el modelo"] == "Sí",
    "Variable"
].tolist()

variables_finales = [
    var for var in variables_finales
    if var != "percentil_talla"
]

base_modelado = df[
    variables_finales + ["rendimiento_clase", "rendimiento_binario"]
].copy()

print("Filas:", base_modelado.shape[0])
print("Variables predictoras finales:", len(variables_finales))
print("Columnas totales:", base_modelado.shape[1])

base_modelado.head()

In [ ]:
X = base_modelado.drop(
    columns=["rendimiento_clase", "rendimiento_binario"],
    errors="ignore"
)

y = base_modelado["rendimiento_binario"]

print("X:", X.shape)
print("y:", y.shape)
print(y.value_counts())

In [ ]:
variables_numericas = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

variables_categoricas = X.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Variables numéricas:", len(variables_numericas))
print("Variables categóricas:", len(variables_categoricas))
print("Variables categóricas:", variables_categoricas)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("\nDistribución entrenamiento:")
print(y_train.value_counts(normalize=True) * 100)

print("\nDistribución prueba:")
print(y_test.value_counts(normalize=True) * 100)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocesamiento = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline(steps=[
                ("imputacion", SimpleImputer(strategy="median")),
                ("escalamiento", StandardScaler())
            ]),
            variables_numericas
        )
    ],
    remainder="drop"
)

preprocesamiento

In [ ]:
import numpy as np
import pandas as pd

# Calcular matriz de correlación de Pearson
matriz_pearson = X.corr(method="pearson")

# Definir umbral de alta correlación
umbral_correlacion = 0.90

# Tomar valores absolutos
corr_abs = matriz_pearson.abs()

# Usar solo el triángulo superior para evitar duplicados
triangulo_superior = corr_abs.where(
    np.triu(np.ones(corr_abs.shape), k=1).astype(bool)
)

# Convertir pares correlacionados a tabla
pares_correlacionados = (
    triangulo_superior
    .stack()
    .reset_index()
)

pares_correlacionados.columns = [
    "Variable 1",
    "Variable 2",
    "Correlación absoluta"
]

# Filtrar pares con correlación alta
pares_correlacionados = pares_correlacionados[
    pares_correlacionados["Correlación absoluta"] >= umbral_correlacion
].sort_values("Correlación absoluta", ascending=False)

print("Número de pares altamente correlacionados:", pares_correlacionados.shape[0])
print("Correlación máxima:", pares_correlacionados["Correlación absoluta"].max())

pares_correlacionados.head(20)

In [ ]:
tabla_pearson_articulo = pares_correlacionados.head(10).copy()

tabla_pearson_articulo["Variable 1"] = (
    tabla_pearson_articulo["Variable 1"]
    .str.replace("_", " ", regex=False)
    .str.capitalize()
)

tabla_pearson_articulo["Variable 2"] = (
    tabla_pearson_articulo["Variable 2"]
    .str.replace("_", " ", regex=False)
    .str.capitalize()
)

tabla_pearson_articulo["Correlación absoluta"] = (
    tabla_pearson_articulo["Correlación absoluta"]
    .round(3)
)

tabla_pearson_articulo

In [ ]:
top_correlaciones = pares_correlacionados.head(8).copy()

top_correlaciones["Par"] = [
    f"{v1.replace('_', ' ')}\n{v2.replace('_', ' ')}"
    for v1, v2 in zip(top_correlaciones["Variable 1"], top_correlaciones["Variable 2"])
]

top_correlaciones = top_correlaciones.sort_values(
    "Correlación absoluta",
    ascending=True
)

plt.figure(figsize=(8, 5.8), dpi=150)

sns.barplot(
    data=top_correlaciones,
    x="Correlación absoluta",
    y="Par",
    color="#5E6A71"
)

plt.title(
    "Pares morfológicos con mayor correlación de Pearson",
    fontsize=12,
    weight="bold"
)

plt.xlabel("Correlación absoluta")
plt.ylabel("")
plt.xlim(0.98, 1.01)

for index, value in enumerate(top_correlaciones["Correlación absoluta"]):
    plt.text(
        value + 0.001,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

plt.yticks(fontsize=8)
sns.despine(left=True, bottom=False)
plt.tight_layout()

plt.savefig(
    "figura_pearson_top8_revista.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Tomar los 20 pares más correlacionados
top_pares = pares_correlacionados.head(20).copy()

# Obtener variables únicas que aparecen en esos pares
variables_top_correlacion = pd.unique(
    top_pares[["Variable 1", "Variable 2"]].values.ravel()
).tolist()

# Si salen demasiadas, limitar a las primeras 20
variables_top_correlacion = variables_top_correlacion[:20]

# Crear matriz reducida
matriz_pearson_reducida = X[variables_top_correlacion].corr(method="pearson")

# Nombres más legibles
nombres_legibles = [
    var.replace("_", " ").capitalize()
    for var in variables_top_correlacion
]

plt.figure(figsize=(13, 11), dpi=200)

sns.heatmap(
    matriz_pearson_reducida,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Coeficiente de Pearson"},
    xticklabels=nombres_legibles,
    yticklabels=nombres_legibles,
    annot_kws={"size": 7}
)

plt.title(
    "Matriz de correlación de Pearson de variables morfológicas seleccionadas",
    fontsize=14,
    weight="bold",
    pad=14
)

plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)

plt.tight_layout()

plt.savefig(
    "figura_matriz_pearson_reducida_revista.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef
)

!pip install -q xgboost
from xgboost import XGBClassifier

modelos_ml = {
    "Regresión Logística": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )
}

valores_k = [20, 30, 40, 50, 60, 70, 91]

resultados_k = []

for k in valores_k:

    selector_k = SelectKBest(
        score_func=f_classif,
        k=k
    )

    preprocesamiento_k = Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("seleccion_caracteristicas", selector_k)
    ])

    for nombre_modelo, modelo in modelos_ml.items():

        pipeline = Pipeline(steps=[
            ("preprocesamiento_y_seleccion", preprocesamiento_k),
            ("modelo", modelo)
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        resultados_k.append({
            "k_variables": k,
            "Modelo": nombre_modelo,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
            "Precisión": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1-score": f1_score(y_test, y_pred, zero_division=0),
            "AUC-ROC": roc_auc_score(y_test, y_prob),
            "MCC": matthews_corrcoef(y_test, y_pred)
        })

resultados_k = pd.DataFrame(resultados_k)

resultados_k_ordenado = resultados_k.sort_values(
    ["F1-score", "Balanced accuracy", "AUC-ROC"],
    ascending=False
)

resultados_k_ordenado.head(15)

In [ ]:
plt.figure(figsize=(10, 5.5), dpi=150)

sns.lineplot(
    data=resultados_k,
    x="k_variables",
    y="F1-score",
    hue="Modelo",
    marker="o"
)

plt.title(
    "Evaluación de SelectKBest según número de variables conservadas",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Número de variables conservadas (k)")
plt.ylabel("F1-score")
plt.ylim(0, 1)

plt.legend(title="")
sns.despine(left=False, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_selectkbest_comparacion_k.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

# Configuración final: k = 91
k_final = 91

selector_final = SelectKBest(
    score_func=f_classif,
    k=k_final
)

preprocesamiento_final = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("seleccion_caracteristicas", selector_final)
])

resultados_ml_final = []
matrices_ml_final = {}
predicciones_ml_final = {}
probabilidades_ml_final = {}
pipelines_ml_final = {}

for nombre_modelo, modelo in modelos_ml.items():

    pipeline = Pipeline(steps=[
        ("preprocesamiento_y_seleccion", preprocesamiento_final),
        ("modelo", modelo)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    matriz = confusion_matrix(y_test, y_pred)

    resultados_ml_final.append({
        "Modelo": nombre_modelo,
        "k_variables": k_final,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precisión": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_test, y_prob),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

    matrices_ml_final[nombre_modelo] = matriz
    predicciones_ml_final[nombre_modelo] = y_pred
    probabilidades_ml_final[nombre_modelo] = y_prob
    pipelines_ml_final[nombre_modelo] = pipeline

resultados_ml_final = pd.DataFrame(resultados_ml_final)

resultados_ml_final = resultados_ml_final.sort_values(
    ["F1-score", "Balanced accuracy", "AUC-ROC"],
    ascending=False
)

resultados_ml_final.round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 8), dpi=150)
axes = axes.flatten()

orden_modelos_ml = ["Regresión Logística", "SVM", "Random Forest", "XGBoost"]

for ax, modelo in zip(axes, orden_modelos_ml):

    matriz = matrices_ml_final[modelo]

    sns.heatmap(
        matriz,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["En desarrollo", "Alto rendimiento"],
        yticklabels=["En desarrollo", "Alto rendimiento"],
        annot_kws={"size": 13, "weight": "bold"},
        ax=ax
    )

    ax.set_title(modelo, fontsize=11, weight="bold")
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Valor real")

plt.suptitle(
    "Matrices de confusión de modelos de Machine Learning",
    fontsize=13,
    weight="bold",
    y=1.02
)

plt.tight_layout()

plt.savefig(
    "figura_matrices_confusion_modelos_ml_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 8), dpi=180)
axes = axes.flatten()

orden_modelos_ml = ["Regresión Logística", "SVM", "Random Forest", "XGBoost"]

for ax, modelo in zip(axes, orden_modelos_ml):

    matriz = matrices_ml_final[modelo]

    sns.heatmap(
        matriz,
        annot=True,
        fmt="d",
        cmap=sns.light_palette("#4B6F8C", as_cmap=True),
        cbar=False,
        xticklabels=["En desarrollo", "Alto rendimiento"],
        yticklabels=["En desarrollo", "Alto rendimiento"],
        annot_kws={"size": 16, "weight": "bold"},
        linewidths=0.8,
        linecolor="white",
        ax=ax
    )

    ax.set_title(modelo, fontsize=12, weight="bold", pad=10)
    ax.set_xlabel("Predicción", fontsize=10)
    ax.set_ylabel("Valor real", fontsize=10)
    ax.tick_params(axis="x", labelsize=9, rotation=0)
    ax.tick_params(axis="y", labelsize=9, rotation=90)

plt.suptitle(
    "Matrices de confusión de los modelos de Machine Learning",
    fontsize=14,
    weight="bold",
    y=1.03
)

plt.tight_layout(pad=2.0)

plt.savefig(
    "figura_matrices_confusion_modelos_ml_final_revista.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
tabla_metricas_ml_final = resultados_ml_final.copy()

columnas_metricas = [
    "Accuracy",
    "Balanced accuracy",
    "Precisión",
    "Recall",
    "F1-score",
    "AUC-ROC",
    "MCC"
]

tabla_metricas_ml_final[columnas_metricas] = (
    tabla_metricas_ml_final[columnas_metricas].round(3)
)

tabla_metricas_ml_final.to_excel(
    "tabla_metricas_modelos_ml_final.xlsx",
    index=False
)

files.download("tabla_metricas_modelos_ml_final.xlsx")

tabla_metricas_ml_final

In [ ]:
metricas_grafico_ml = tabla_metricas_ml_final.melt(
    id_vars="Modelo",
    value_vars=["Accuracy", "Balanced accuracy", "F1-score", "AUC-ROC"],
    var_name="Métrica",
    value_name="Valor"
)

plt.figure(figsize=(10, 5.5), dpi=150)

sns.barplot(
    data=metricas_grafico_ml,
    x="Modelo",
    y="Valor",
    hue="Métrica",
    palette=["#4B6F8C", "#7C8A92", "#A7B0B5", "#C7A76C"]
)

plt.title(
    "Comparación del desempeño de modelos de Machine Learning",
    fontsize=13,
    weight="bold"
)

plt.xlabel("")
plt.ylabel("Valor de la métrica")
plt.ylim(0, 1)

plt.xticks(rotation=15, ha="right")
plt.legend(title="", loc="lower right", frameon=True)

sns.despine(left=False, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_comparacion_metricas_modelos_ml_final.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Asegurar orden
resultados_k_plot = resultados_k.copy()
resultados_k_plot = resultados_k_plot.sort_values(["Modelo", "k_variables"])

plt.figure(figsize=(9.5, 5.5), dpi=150)

sns.lineplot(
    data=resultados_k_plot,
    x="k_variables",
    y="F1-score",
    hue="Modelo",
    marker="o",
    linewidth=2
)

plt.title(
    "Evaluación de SelectKBest según número de variables conservadas",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Número de variables conservadas (k)", fontsize=11)
plt.ylabel("F1-score", fontsize=11)
plt.ylim(0, 1)
plt.xticks([20, 30, 40, 50, 60, 70, 91])

plt.legend(
    title="",
    loc="lower right",
    frameon=True,
    fontsize=9
)

sns.despine(left=False, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_evaluacion_selectkbest_k.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
palette_modelos = {
    "Regresión Logística": "#2F4F5F",
    "SVM": "#7C8A92",
    "Random Forest": "#A7B0B5",
    "XGBoost": "#C7A76C"
}

plt.figure(figsize=(9.5, 5.5), dpi=150)

sns.lineplot(
    data=resultados_k_plot,
    x="k_variables",
    y="F1-score",
    hue="Modelo",
    marker="o",
    linewidth=2,
    palette=palette_modelos
)

plt.title(
    "Evaluación de SelectKBest según número de variables conservadas",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Número de variables conservadas (k)", fontsize=11)
plt.ylabel("F1-score", fontsize=11)
plt.ylim(0, 1)
plt.xticks([20, 30, 40, 50, 60, 70, 91])

plt.legend(
    title="",
    loc="lower right",
    frameon=True,
    fontsize=9
)

sns.despine(left=False, bottom=False)
plt.tight_layout()

plt.savefig(
    "figura_evaluacion_selectkbest_k_revista.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

metricas_cv = {
    "Accuracy": "accuracy",
    "Precisión": "precision",
    "Recall": "recall",
    "F1-score": "f1",
    "AUC-ROC": "roc_auc"
}

resultados_cv_simple = []

for nombre_modelo, modelo in modelos_ml.items():

    pipeline_cv = Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", modelo)
    ])

    scores = cross_validate(
        pipeline_cv,
        X,
        y,
        cv=cv5,
        scoring=metricas_cv,
        return_train_score=False
    )

    resultados_cv_simple.append({
        "Modelo": nombre_modelo,
        "Accuracy": scores["test_Accuracy"].mean(),
        "Precisión": scores["test_Precisión"].mean(),
        "Recall": scores["test_Recall"].mean(),
        "F1-score": scores["test_F1-score"].mean(),
        "AUC-ROC": scores["test_AUC-ROC"].mean()
    })

resultados_cv_simple = pd.DataFrame(resultados_cv_simple)

resultados_cv_simple = resultados_cv_simple.sort_values(
    "F1-score",
    ascending=False
)

resultados_cv_simple.round(3)

In [ ]:
resultados_cv_formato = []

for nombre_modelo, modelo in modelos_ml.items():

    pipeline_cv = Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", modelo)
    ])

    scores = cross_validate(
        pipeline_cv,
        X,
        y,
        cv=cv5,
        scoring=metricas_cv,
        return_train_score=False
    )

    resultados_cv_formato.append({
        "Modelo": nombre_modelo,
        "Accuracy": f"{scores['test_Accuracy'].mean():.3f} ± {scores['test_Accuracy'].std():.3f}",
        "Precisión": f"{scores['test_Precisión'].mean():.3f} ± {scores['test_Precisión'].std():.3f}",
        "Recall": f"{scores['test_Recall'].mean():.3f} ± {scores['test_Recall'].std():.3f}",
        "F1-score": f"{scores['test_F1-score'].mean():.3f} ± {scores['test_F1-score'].std():.3f}",
        "AUC-ROC": f"{scores['test_AUC-ROC'].mean():.3f} ± {scores['test_AUC-ROC'].std():.3f}"
    })

resultados_cv_formato = pd.DataFrame(resultados_cv_formato)
resultados_cv_formato

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

tabla_cv_plot = resultados_cv_simple.copy()
tabla_cv_plot = tabla_cv_plot.sort_values("F1-score", ascending=True)

plt.figure(figsize=(8.5, 4.8), dpi=150)

sns.barplot(
    data=tabla_cv_plot,
    x="F1-score",
    y="Modelo",
    color="#5E6A71"
)

plt.title(
    "Validación cruzada estratificada de 5 pliegues",
    fontsize=13,
    weight="bold"
)

plt.xlabel("F1-score promedio")
plt.ylabel("")
plt.xlim(0, 1)

for index, value in enumerate(tabla_cv_plot["F1-score"]):
    plt.text(
        value + 0.01,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=10
    )

sns.despine(left=True, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_validacion_cruzada_5fold_f1.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Ajustar el preprocesamiento con los datos de entrenamiento
X_train_dl = preprocesamiento.fit_transform(X_train)
X_test_dl = preprocesamiento.transform(X_test)

print("X_train_dl:", X_train_dl.shape)
print("X_test_dl:", X_test_dl.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Conv1D,
    GlobalAveragePooling1D,
    MultiHeadAttention,
    LayerNormalization,
    Add,
    Flatten
)
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
from sklearn.metrics import confusion_matrix

def evaluar_modelo_dl(nombre_modelo, modelo, X_eval, y_eval):
    y_prob = modelo.predict(X_eval).ravel()
    y_pred = (y_prob >= 0.50).astype(int)

    matriz = confusion_matrix(y_eval, y_pred)

    metricas = {
        "Modelo": nombre_modelo,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_eval, y_pred),
        "Precisión": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1-score": f1_score(y_eval, y_pred, zero_division=0),
        "AUC-ROC": roc_auc_score(y_eval, y_prob),
        "MCC": matthews_corrcoef(y_eval, y_pred)
    }

    return metricas, matriz, y_pred, y_prob

In [ ]:
modelo_mlp = Sequential([
    Input(shape=(X_train_dl.shape[1],)),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.30),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid")
])

modelo_mlp.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

historial_mlp = modelo_mlp.fit(
    X_train_dl,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=8,
    callbacks=[early_stop],
    verbose=0
)

metricas_mlp, matriz_mlp, y_pred_mlp, y_prob_mlp = evaluar_modelo_dl(
    "MLP",
    modelo_mlp,
    X_test_dl,
    y_test
)

pd.DataFrame([metricas_mlp]).round(3)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5.5, 4.5), dpi=150)

sns.heatmap(
    matriz_mlp,
    annot=True,
    fmt="d",
    cmap=sns.light_palette("#4B6F8C", as_cmap=True),
    cbar=False,
    xticklabels=["En desarrollo", "Alto rendimiento"],
    yticklabels=["En desarrollo", "Alto rendimiento"],
    annot_kws={"size": 16, "weight": "bold"},
    linewidths=0.8,
    linecolor="white"
)

plt.title(
    "Matriz de confusión del modelo MLP",
    fontsize=12,
    weight="bold",
    pad=10
)

plt.xlabel("Predicción", fontsize=10)
plt.ylabel("Valor real", fontsize=10)

plt.xticks(fontsize=9)
plt.yticks(fontsize=9, rotation=90)

plt.tight_layout()

plt.savefig(
    "figura_matriz_confusion_mlp.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

matriz_mlp

In [ ]:
X_train_cnn = X_train_dl.reshape(X_train_dl.shape[0], X_train_dl.shape[1], 1)
X_test_cnn = X_test_dl.reshape(X_test_dl.shape[0], X_test_dl.shape[1], 1)

print("X_train_cnn:", X_train_cnn.shape)
print("X_test_cnn:", X_test_cnn.shape)

In [ ]:
modelo_cnn1d = Sequential([
    Input(shape=(X_train_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=3, activation="relu", padding="same"),
    BatchNormalization(),
    Dropout(0.30),
    Conv1D(filters=16, kernel_size=3, activation="relu", padding="same"),
    GlobalAveragePooling1D(),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(1, activation="sigmoid")
])

modelo_cnn1d.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop_cnn = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

historial_cnn1d = modelo_cnn1d.fit(
    X_train_cnn,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=8,
    callbacks=[early_stop_cnn],
    verbose=0
)

metricas_cnn1d, matriz_cnn1d, y_pred_cnn1d, y_prob_cnn1d = evaluar_modelo_dl(
    "CNN1D",
    modelo_cnn1d,
    X_test_cnn,
    y_test
)

pd.DataFrame([metricas_cnn1d]).round(3)

In [ ]:
plt.figure(figsize=(5.5, 4.5), dpi=150)

sns.heatmap(
    matriz_cnn1d,
    annot=True,
    fmt="d",
    cmap=sns.light_palette("#4B6F8C", as_cmap=True),
    cbar=False,
    xticklabels=["En desarrollo", "Alto rendimiento"],
    yticklabels=["En desarrollo", "Alto rendimiento"],
    annot_kws={"size": 16, "weight": "bold"},
    linewidths=0.8,
    linecolor="white"
)

plt.title(
    "Matriz de confusión del modelo CNN1D",
    fontsize=12,
    weight="bold",
    pad=10
)

plt.xlabel("Predicción", fontsize=10)
plt.ylabel("Valor real", fontsize=10)

plt.xticks(fontsize=9)
plt.yticks(fontsize=9, rotation=90)

plt.tight_layout()

plt.savefig(
    "figura_matriz_confusion_cnn1d.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

matriz_cnn1d

In [ ]:
# Entrada para modelo Attention
entrada = Input(shape=(X_train_cnn.shape[1], 1))

# Proyección inicial
x = Dense(32)(entrada)

# Bloque de atención
attn = MultiHeadAttention(
    num_heads=2,
    key_dim=16
)(x, x)

x = Add()([x, attn])
x = LayerNormalization()(x)

# Capa densa final
x = GlobalAveragePooling1D()(x)
x = Dense(32, activation="relu")(x)
x = Dropout(0.30)(x)
salida = Dense(1, activation="sigmoid")(x)

modelo_attention = Model(
    inputs=entrada,
    outputs=salida
)

modelo_attention.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop_attention = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

historial_attention = modelo_attention.fit(
    X_train_cnn,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=8,
    callbacks=[early_stop_attention],
    verbose=0
)

metricas_attention, matriz_attention, y_pred_attention, y_prob_attention = evaluar_modelo_dl(
    "Attention",
    modelo_attention,
    X_test_cnn,
    y_test
)

pd.DataFrame([metricas_attention]).round(3)

In [ ]:
plt.figure(figsize=(5.5, 4.5), dpi=150)

sns.heatmap(
    matriz_attention,
    annot=True,
    fmt="d",
    cmap=sns.light_palette("#4B6F8C", as_cmap=True),
    cbar=False,
    xticklabels=["En desarrollo", "Alto rendimiento"],
    yticklabels=["En desarrollo", "Alto rendimiento"],
    annot_kws={"size": 16, "weight": "bold"},
    linewidths=0.8,
    linecolor="white"
)

plt.title(
    "Matriz de confusión del modelo Attention",
    fontsize=12,
    weight="bold",
    pad=10
)

plt.xlabel("Predicción", fontsize=10)
plt.ylabel("Valor real", fontsize=10)

plt.xticks(fontsize=9)
plt.yticks(fontsize=9, rotation=90)

plt.tight_layout()

plt.savefig(
    "figura_matriz_confusion_attention.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

matriz_attention

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight = {
    0: pesos[0],
    1: pesos[1]
}

class_weight

In [ ]:
# Modelo Attention con pesos de clase
entrada = Input(shape=(X_train_cnn.shape[1], 1))

x = Dense(32)(entrada)

attn = MultiHeadAttention(
    num_heads=2,
    key_dim=16
)(x, x)

x = Add()([x, attn])
x = LayerNormalization()(x)

x = GlobalAveragePooling1D()(x)
x = Dense(32, activation="relu")(x)
x = Dropout(0.30)(x)
salida = Dense(1, activation="sigmoid")(x)

modelo_attention_balanced = Model(
    inputs=entrada,
    outputs=salida
)

modelo_attention_balanced.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stop_attention_balanced = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

historial_attention_balanced = modelo_attention_balanced.fit(
    X_train_cnn,
    y_train,
    validation_split=0.20,
    epochs=150,
    batch_size=8,
    callbacks=[early_stop_attention_balanced],
    class_weight=class_weight,
    verbose=0
)

metricas_attention_balanced, matriz_attention_balanced, y_pred_attention_balanced, y_prob_attention_balanced = evaluar_modelo_dl(
    "Attention balanceado",
    modelo_attention_balanced,
    X_test_cnn,
    y_test
)

pd.DataFrame([metricas_attention_balanced]).round(3)

In [ ]:
plt.figure(figsize=(5.5, 4.5), dpi=150)

sns.heatmap(
    matriz_attention_balanced,
    annot=True,
    fmt="d",
    cmap=sns.light_palette("#4B6F8C", as_cmap=True),
    cbar=False,
    xticklabels=["En desarrollo", "Alto rendimiento"],
    yticklabels=["En desarrollo", "Alto rendimiento"],
    annot_kws={"size": 16, "weight": "bold"},
    linewidths=0.8,
    linecolor="white"
)

plt.title(
    "Matriz de confusión del modelo Attention balanceado",
    fontsize=12,
    weight="bold",
    pad=10
)

plt.xlabel("Predicción", fontsize=10)
plt.ylabel("Valor real", fontsize=10)

plt.xticks(fontsize=9)
plt.yticks(fontsize=9, rotation=90)

plt.tight_layout()

plt.savefig(
    "figura_matriz_confusion_attention_balanceado.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

matriz_attention_balanced

In [ ]:
resultados_dl = pd.DataFrame([
    metricas_mlp,
    metricas_cnn1d,
    metricas_attention
])

resultados_dl = resultados_dl.sort_values(
    ["F1-score", "Balanced accuracy", "AUC-ROC"],
    ascending=False
)

tabla_metricas_dl = resultados_dl.round(3)

tabla_metricas_dl

In [ ]:
matrices_dl = {
    "MLP": matriz_mlp,
    "CNN1D": matriz_cnn1d,
    "Attention": matriz_attention
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=150)

for ax, (modelo, matriz) in zip(axes, matrices_dl.items()):

    sns.heatmap(
        matriz,
        annot=True,
        fmt="d",
        cmap=sns.light_palette("#4B6F8C", as_cmap=True),
        cbar=False,
        xticklabels=["En desarrollo", "Alto rendimiento"],
        yticklabels=["En desarrollo", "Alto rendimiento"],
        annot_kws={"size": 15, "weight": "bold"},
        linewidths=0.8,
        linecolor="white",
        ax=ax
    )

    ax.set_title(modelo, fontsize=12, weight="bold", pad=10)
    ax.set_xlabel("Predicción", fontsize=10)
    ax.set_ylabel("Valor real", fontsize=10)
    ax.tick_params(axis="x", labelsize=8, rotation=0)
    ax.tick_params(axis="y", labelsize=8, rotation=90)

plt.suptitle(
    "Matrices de confusión de modelos de Deep Learning",
    fontsize=14,
    weight="bold",
    y=1.05
)

plt.tight_layout()

plt.savefig(
    "figura_matrices_confusion_modelos_dl.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Preparar tabla ML final
tabla_ml_final = resultados_ml_final.copy()
tabla_ml_final["Tipo de modelo"] = "Machine Learning"

# Quitar columna k_variables para poder unir con DL
tabla_ml_final = tabla_ml_final.drop(columns=["k_variables"], errors="ignore")

# Preparar tabla DL final
tabla_dl_final = resultados_dl.copy()
tabla_dl_final["Tipo de modelo"] = "Deep Learning"

# Unir ML + DL
tabla_comparacion_final = pd.concat(
    [tabla_ml_final, tabla_dl_final],
    ignore_index=True
)

tabla_comparacion_final = tabla_comparacion_final.sort_values(
    ["F1-score", "Balanced accuracy", "AUC-ROC"],
    ascending=False
)

columnas_metricas = [
    "Accuracy",
    "Balanced accuracy",
    "Precisión",
    "Recall",
    "F1-score",
    "AUC-ROC",
    "MCC"
]

tabla_comparacion_final[columnas_metricas] = (
    tabla_comparacion_final[columnas_metricas].round(3)
)

tabla_comparacion_final

In [ ]:
tabla_comparacion_final.to_excel(
    "tabla_comparacion_final_7_modelos.xlsx",
    index=False
)

files.download("tabla_comparacion_final_7_modelos.xlsx")

In [ ]:
metricas_final_grafico = tabla_comparacion_final.melt(
    id_vars=["Modelo", "Tipo de modelo"],
    value_vars=["F1-score", "Balanced accuracy"],
    var_name="Métrica",
    value_name="Valor"
)

plt.figure(figsize=(10, 5.8), dpi=150)

sns.barplot(
    data=metricas_final_grafico,
    x="Modelo",
    y="Valor",
    hue="Métrica",
    palette=["#4B6F8C", "#C7A76C"]
)

plt.title(
    "Comparación final de modelos ML y DL",
    fontsize=13,
    weight="bold"
)

plt.xlabel("")
plt.ylabel("Valor de la métrica")
plt.ylim(0, 1)

plt.xticks(rotation=20, ha="right")
plt.legend(title="", loc="lower right", frameon=True)

sns.despine(left=False, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_comparacion_final_modelos_ml_dl.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
metricas_final_grafico = tabla_comparacion_final.melt(
    id_vars=["Modelo", "Tipo de modelo"],
    value_vars=["F1-score", "Balanced accuracy"],
    var_name="Métrica",
    value_name="Valor"
)

plt.figure(figsize=(10, 5.8), dpi=150)

sns.barplot(
    data=metricas_final_grafico,
    x="Modelo",
    y="Valor",
    hue="Métrica",
    palette=["#4B6F8C", "#C7A76C"]
)

plt.title(
    "Comparación final de modelos ML y DL",
    fontsize=13,
    weight="bold"
)

plt.xlabel("")
plt.ylabel("Valor de la métrica")
plt.ylim(0, 1)

plt.xticks(rotation=20, ha="right")
plt.legend(title="", loc="lower right", frameon=True)

sns.despine(left=False, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_comparacion_final_f1_balanced_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Seleccionar pipeline de Regresión Logística
pipeline_logistica = pipelines_ml_final["Regresión Logística"]

# Obtener nombres de variables después del preprocesamiento
nombres_variables = variables_numericas.copy()

# Obtener coeficientes del modelo
coeficientes = pipeline_logistica.named_steps["modelo"].coef_[0]

importancia_logistica = pd.DataFrame({
    "Variable": nombres_variables,
    "Coeficiente": coeficientes
})

importancia_logistica["Importancia absoluta"] = importancia_logistica["Coeficiente"].abs()

importancia_logistica = importancia_logistica.sort_values(
    "Importancia absoluta",
    ascending=False
)

importancia_logistica.head(15)

In [ ]:
importancia_logistica_dim = importancia_logistica.merge(
    tabla_variables[["Variable", "Dimensión morfológica"]],
    on="Variable",
    how="left"
)

importancia_logistica_dim.head(15)

In [ ]:
top_importancia = importancia_logistica_dim.head(15).copy()

top_importancia["Variable legible"] = (
    top_importancia["Variable"]
    .str.replace("_", " ", regex=False)
    .str.capitalize()
)

top_importancia = top_importancia.sort_values(
    "Importancia absoluta",
    ascending=True
)

plt.figure(figsize=(9, 6), dpi=150)

sns.barplot(
    data=top_importancia,
    x="Importancia absoluta",
    y="Variable legible",
    color="#5E6A71"
)

plt.title(
    "Variables morfológicas más influyentes en Regresión Logística",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Importancia absoluta del coeficiente")
plt.ylabel("")

for index, value in enumerate(top_importancia["Importancia absoluta"]):
    plt.text(
        value + 0.01,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

sns.despine(left=True, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_importancia_variables_regresion_logistica.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
importancia_logistica_dim.head(15)[
    ["Variable", "Dimensión morfológica", "Coeficiente", "Importancia absoluta"]
]

In [ ]:
# Diccionario de nombres completos para variables abreviadas

nombres_completos_variables = {
    "adip_muslo": "Tejido adiposo del muslo",
    "musc_muslo": "Masa muscular del muslo",
    "pantmax": "Perímetro máximo de pantorrilla",
    "radial_estiloidea": "Longitud radial-estiloidea",
    "masa_piel_kg": "Masa de piel (kg)",
    "area_total_muslo": "Área total del muslo",
    "masa_lipida_grasa": "Masa lipídica grasa",
    "porcentaje_grasa_del_brazo": "% grasa del brazo",
    "indice_braquial_ibb": "Índice braquial",
    "indice_biacromial_biliocrestal": "Índice biacromial-biliocrestal",
    "indice_acromio_iliaco": "Índice acromio-ilíaco"
}

def nombre_legible_variable(variable):
    variable_lower = variable.lower()

    if variable_lower in nombres_completos_variables:
        return nombres_completos_variables[variable_lower]

    nombre = variable.replace("_", " ")

    reemplazos = {
        "porcentaje": "%",
        "indice": "Índice",
        "masa": "Masa",
        "lipida": "lipídica",
        "osea": "ósea",
        "adiposa": "adiposa",
        "muscular": "muscular",
        "morfologica": "morfológica",
        "biologica": "biológica",
        "maduracion": "maduración",
        "envergadura": "envergadura",
        "talla": "talla",
        "peso": "peso",
        "grasa": "grasa",
        "piel": "piel"
    }

    for original, nuevo in reemplazos.items():
        nombre = nombre.replace(original, nuevo)

    return nombre.capitalize()

In [ ]:
top_importancia = importancia_logistica_dim.head(15).copy()

top_importancia["Variable legible"] = top_importancia["Variable"].apply(
    nombre_legible_variable
)

top_importancia = top_importancia.sort_values(
    "Importancia absoluta",
    ascending=True
)

plt.figure(figsize=(9.5, 6.5), dpi=150)

sns.barplot(
    data=top_importancia,
    x="Importancia absoluta",
    y="Variable legible",
    color="#5E6A71"
)

plt.title(
    "Variables morfológicas más influyentes en Regresión Logística",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Importancia absoluta del coeficiente")
plt.ylabel("")

for index, value in enumerate(top_importancia["Importancia absoluta"]):
    plt.text(
        value + 0.01,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

plt.yticks(fontsize=9)
sns.despine(left=True, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_variables_influyentes_regresion_logistica_nombres_reales.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
top_coeficientes = importancia_logistica_dim.head(15).copy()

top_coeficientes["Variable legible"] = top_coeficientes["Variable"].apply(
    nombre_legible_variable
)

top_coeficientes["Dirección"] = np.where(
    top_coeficientes["Coeficiente"] > 0,
    "Favorece Alto rendimiento",
    "Favorece En desarrollo"
)

top_coeficientes[
    [
        "Variable legible",
        "Dimensión morfológica",
        "Coeficiente",
        "Importancia absoluta",
        "Dirección"
    ]
].round(3)

In [ ]:
top_direccion = top_coeficientes.copy()

top_direccion = top_direccion.sort_values(
    "Coeficiente",
    ascending=True
)

plt.figure(figsize=(9.5, 6.5), dpi=150)

colores = top_direccion["Dirección"].map({
    "Favorece Alto rendimiento": "#4B6F8C",
    "Favorece En desarrollo": "#C7A76C"
})

plt.barh(
    top_direccion["Variable legible"],
    top_direccion["Coeficiente"],
    color=colores
)

plt.axvline(0, color="#333333", linewidth=1)

plt.title(
    "Dirección de influencia de variables morfológicas",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Coeficiente de Regresión Logística")
plt.ylabel("")

for index, value in enumerate(top_direccion["Coeficiente"]):
    desplazamiento = 0.03 if value > 0 else -0.03
    alineacion = "left" if value > 0 else "right"

    plt.text(
        value + desplazamiento,
        index,
        f"{value:.3f}",
        va="center",
        ha=alineacion,
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    "figura_direccion_influencia_regresion_logistica.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
top_direccion = top_coeficientes.copy()

top_direccion = top_direccion.sort_values(
    "Coeficiente",
    ascending=True
)

plt.figure(figsize=(10.5, 6.8), dpi=150)

colores = top_direccion["Dirección"].map({
    "Favorece Alto rendimiento": "#4B6F8C",
    "Favorece En desarrollo": "#C7A76C"
})

plt.barh(
    top_direccion["Variable legible"],
    top_direccion["Coeficiente"],
    color=colores
)

plt.axvline(0, color="#333333", linewidth=1)

plt.title(
    "Dirección de influencia de variables morfológicas",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Coeficiente de Regresión Logística")
plt.ylabel("")

# Dejar más espacio para etiquetas negativas
plt.xlim(-1.35, 1.65)

for index, value in enumerate(top_direccion["Coeficiente"]):
    if value > 0:
        plt.text(
            value + 0.04,
            index,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=9
        )
    else:
        plt.text(
            value - 0.04,
            index,
            f"{value:.3f}",
            va="center",
            ha="right",
            fontsize=9
        )

plt.yticks(fontsize=9)

sns.despine(left=False, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_direccion_influencia_regresion_logistica_revista.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
!pip install -q shap

import shap

In [ ]:
# Pipeline ganador
pipeline_logistica = pipelines_ml_final["Regresión Logística"]

# Extraer preprocesador y modelo
preprocesador_logistica = pipeline_logistica.named_steps["preprocesamiento_y_seleccion"]
modelo_logistica = pipeline_logistica.named_steps["modelo"]

# Transformar datos
X_train_proc = preprocesador_logistica.transform(X_train)
X_test_proc = preprocesador_logistica.transform(X_test)

# Nombres de variables
nombres_variables_proc = variables_numericas.copy()

print("X_train_proc:", X_train_proc.shape)
print("X_test_proc:", X_test_proc.shape)
print("Variables:", len(nombres_variables_proc))

In [ ]:
explainer = shap.LinearExplainer(
    modelo_logistica,
    X_train_proc
)

shap_values = explainer.shap_values(X_test_proc)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_proc,
    feature_names=nombres_variables_proc,
    plot_type="bar",
    show=False,
    max_display=15
)

plt.title(
    "Importancia global de variables mediante SHAP",
    fontsize=13,
    weight="bold"
)

plt.tight_layout()

plt.savefig(
    "figura_shap_importancia_global_regresion_logistica.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
importancia_shap = pd.DataFrame({
    "Variable": nombres_variables_proc,
    "SHAP absoluto medio": np.abs(shap_values).mean(axis=0)
})

importancia_shap = importancia_shap.sort_values(
    "SHAP absoluto medio",
    ascending=False
)

importancia_shap.head(15)

In [ ]:
importancia_shap["Variable legible"] = importancia_shap["Variable"].apply(
    nombre_legible_variable
)

importancia_shap.head(15)[
    ["Variable legible", "SHAP absoluto medio"]
]

In [ ]:
top_shap = importancia_shap.head(15).copy()

top_shap = top_shap.sort_values(
    "SHAP absoluto medio",
    ascending=True
)

plt.figure(figsize=(9.5, 6.5), dpi=150)

sns.barplot(
    data=top_shap,
    x="SHAP absoluto medio",
    y="Variable legible",
    color="#5E6A71"
)

plt.title(
    "Variables más influyentes según SHAP",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Valor SHAP absoluto medio")
plt.ylabel("")

for index, value in enumerate(top_shap["SHAP absoluto medio"]):
    plt.text(
        value + 0.005,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

sns.despine(left=True, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_shap_variables_influyentes_regresion_logistica.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
top_shap = importancia_shap.head(15).copy()

# Ordenar de menor a mayor
top_shap = top_shap.sort_values(
    "SHAP absoluto medio",
    ascending=True
)

plt.figure(figsize=(9.5, 6.5), dpi=150)

sns.barplot(
    data=top_shap,
    x="SHAP absoluto medio",
    y="Variable legible",
    color="#5E6A71"
)

plt.title(
    "Variables más influyentes según SHAP",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Valor SHAP absoluto medio")
plt.ylabel("")

for index, value in enumerate(top_shap["SHAP absoluto medio"]):
    plt.text(
        value + 0.005,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

sns.despine(left=True, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_shap_variables_influyentes_menor_mayor.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
nombres_completos_variables.update({
    "total_antebr": "Perímetro total del antebrazo",
    "masa_kg": "Masa corporal (kg)"
})

In [ ]:
importancia_shap.head(20)["Variable"].tolist()

In [ ]:
importancia_shap["Variable legible"] = importancia_shap["Variable"].apply(
    nombre_legible_variable
)

In [ ]:
top_shap = importancia_shap.head(15).copy()

top_shap = top_shap.sort_values(
    "SHAP absoluto medio",
    ascending=True
)

plt.figure(figsize=(9.5, 6.5), dpi=150)

sns.barplot(
    data=top_shap,
    x="SHAP absoluto medio",
    y="Variable legible",
    color="#5E6A71"
)

# Invertir eje Y para que el mayor quede arriba
plt.gca().invert_yaxis()

plt.title(
    "Variables más influyentes según SHAP",
    fontsize=13,
    weight="bold"
)

plt.xlabel("Valor SHAP absoluto medio")
plt.ylabel("")

for index, value in enumerate(top_shap["SHAP absoluto medio"]):
    plt.text(
        value + 0.005,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=9
    )

sns.despine(left=True, bottom=False)

plt.tight_layout()

plt.savefig(
    "figura_shap_variables_influyentes_mayor_menor.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# SHAP Summary Plot clásico
shap.summary_plot(
    shap_values,
    X_test_proc,
    feature_names=nombres_variables_proc,
    max_display=15,
    show=False
)

plt.title(
    "SHAP Summary Plot - Importancia de variables",
    fontsize=13,
    weight="bold"
)

plt.tight_layout()

plt.savefig(
    "figura_shap_summary_plot_regresion_logistica.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
nombres_variables_legibles = [
    nombre_legible_variable(var)
    for var in nombres_variables_proc
]

shap.summary_plot(
    shap_values,
    X_test_proc,
    feature_names=nombres_variables_legibles,
    max_display=15,
    show=False
)

plt.title(
    "SHAP Summary Plot - Importancia de variables",
    fontsize=13,
    weight="bold"
)

plt.tight_layout()

plt.savefig(
    "figura_shap_summary_plot_nombres_reales.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def generar_salida_sistema(indice_caso=0):
    # Caso original
    caso_original = X_test.iloc[[indice_caso]].copy()
    y_real_caso = int(y_test.iloc[indice_caso])

    # Predicción del modelo ganador
    prob_alto = pipeline_logistica.predict_proba(caso_original)[0, 1]
    pred_binaria = int(prob_alto >= 0.50)

    clase_real = "Alto rendimiento" if y_real_caso == 1 else "En desarrollo"
    clase_predicha = "Alto rendimiento" if pred_binaria == 1 else "En desarrollo"

    # Preprocesar caso para SHAP
    caso_proc = preprocesador_logistica.transform(caso_original)
    shap_caso = explainer.shap_values(caso_proc)[0]

    # Tabla de contribuciones locales
    contribuciones = pd.DataFrame({
        "Variable": nombres_variables_proc,
        "Valor SHAP": shap_caso
    })

    contribuciones["Importancia absoluta"] = contribuciones["Valor SHAP"].abs()
    contribuciones["Variable legible"] = contribuciones["Variable"].apply(
        nombre_legible_variable
    )

    contribuciones = contribuciones.sort_values(
        "Importancia absoluta",
        ascending=False
    ).head(8)

    contribuciones = contribuciones.sort_values("Valor SHAP", ascending=True)

    # Colores: positivo favorece alto rendimiento, negativo favorece en desarrollo
    colores = contribuciones["Valor SHAP"].apply(
        lambda x: "#4B6F8C" if x > 0 else "#C7A76C"
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 5.5),
        dpi=150,
        gridspec_kw={"width_ratios": [1, 1.4]}
    )

    # Panel izquierdo: resultado
    axes[0].axis("off")

    axes[0].text(
        0.02, 0.92,
        "Salida del sistema",
        fontsize=15,
        weight="bold"
    )

    axes[0].text(
        0.02, 0.78,
        f"Clase real:\n{clase_real}",
        fontsize=12
    )

    axes[0].text(
        0.02, 0.60,
        f"Clase predicha:\n{clase_predicha}",
        fontsize=12,
        weight="bold",
        color="#2F4F5F"
    )

    axes[0].text(
        0.02, 0.42,
        f"Probabilidad de alto rendimiento:\n{prob_alto:.1%}",
        fontsize=12
    )

    # Barra de probabilidad
    axes[0].barh(
        [0],
        [prob_alto],
        color="#4B6F8C",
        height=0.25
    )

    axes[0].barh(
        [0],
        [1 - prob_alto],
        left=[prob_alto],
        color="#D9DEE2",
        height=0.25
    )

    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(-0.5, 1)
    axes[0].axis("off")

    # Panel derecho: contribuciones SHAP
    axes[1].barh(
        contribuciones["Variable legible"],
        contribuciones["Valor SHAP"],
        color=colores
    )

    axes[1].axvline(0, color="#333333", linewidth=1)
    axes[1].set_title(
        "Variables que influyeron en la predicción",
        fontsize=13,
        weight="bold"
    )

    axes[1].set_xlabel("Valor SHAP local")
    axes[1].set_ylabel("")

    for index, value in enumerate(contribuciones["Valor SHAP"]):
        desplazamiento = 0.02 if value > 0 else -0.02
        alineacion = "left" if value > 0 else "right"

        axes[1].text(
            value + desplazamiento,
            index,
            f"{value:.3f}",
            va="center",
            ha=alineacion,
            fontsize=9
        )

    plt.tight_layout()

    nombre_figura = f"salida_interpretativa_sistema_caso_{indice_caso}.png"

    plt.savefig(
        nombre_figura,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return contribuciones

In [ ]:
contribuciones_caso = generar_salida_sistema(indice_caso=0)
contribuciones_caso

In [ ]:
import matplotlib.image as mpimg

def generar_salida_sistema_con_imagen(indice_caso=0, imagen_deportista=None):
    caso_original = X_test.iloc[[indice_caso]].copy()
    y_real_caso = int(y_test.iloc[indice_caso])

    deportista_id = f"Caso {indice_caso + 1:02d}"

    prob_alto = pipeline_logistica.predict_proba(caso_original)[0, 1]
    pred_binaria = int(prob_alto >= 0.50)

    clase_real = "Alto rendimiento" if y_real_caso == 1 else "En desarrollo"
    clase_predicha = "Alto rendimiento" if pred_binaria == 1 else "En desarrollo"

    caso_proc = preprocesador_logistica.transform(caso_original)
    shap_caso = explainer.shap_values(caso_proc)[0]

    contribuciones = pd.DataFrame({
        "Variable": nombres_variables_proc,
        "Valor SHAP": shap_caso
    })

    contribuciones["Importancia absoluta"] = contribuciones["Valor SHAP"].abs()
    contribuciones["Variable legible"] = contribuciones["Variable"].apply(nombre_legible_variable)

    contribuciones = (
        contribuciones
        .sort_values("Importancia absoluta", ascending=False)
        .head(8)
        .sort_values("Valor SHAP", ascending=True)
    )

    colores = contribuciones["Valor SHAP"].apply(
        lambda x: "#4B6F8C" if x > 0 else "#C7A76C"
    )

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5.5),
        dpi=150,
        gridspec_kw={"width_ratios": [0.9, 1.0, 1.5]}
    )

    # Panel 1: imagen deportista
    axes[0].axis("off")

    if imagen_deportista is not None:
        img = mpimg.imread(imagen_deportista)
        axes[0].imshow(img)
        axes[0].set_title("Deportista evaluado", fontsize=13, weight="bold", pad=10)
    else:
        axes[0].text(
            0.5, 0.5,
            "Imagen\ndel deportista",
            ha="center",
            va="center",
            fontsize=14,
            weight="bold",
            color="#5E6A71"
        )

    axes[0].text(
        0.5, -0.08,
        deportista_id,
        ha="center",
        va="center",
        transform=axes[0].transAxes,
        fontsize=12,
        weight="bold"
    )

    # Panel 2: predicción
    axes[1].axis("off")

    axes[1].text(
        0.02, 0.92,
        "Resultado del sistema",
        fontsize=14,
        weight="bold"
    )

    axes[1].text(0.02, 0.74, f"Clase real:\n{clase_real}", fontsize=11)

    axes[1].text(
        0.02, 0.56,
        f"Clase predicha:\n{clase_predicha}",
        fontsize=11,
        weight="bold",
        color="#2F4F5F"
    )

    axes[1].text(
        0.02, 0.38,
        f"Probabilidad de alto rendimiento:\n{prob_alto:.1%}",
        fontsize=11
    )

    axes[1].barh([0], [prob_alto], color="#4B6F8C", height=0.18)
    axes[1].barh([0], [1 - prob_alto], left=[prob_alto], color="#D9DEE2", height=0.18)

    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(-0.5, 1)

    # Panel 3: SHAP local
    axes[2].barh(
        contribuciones["Variable legible"],
        contribuciones["Valor SHAP"],
        color=colores
    )

    axes[2].axvline(0, color="#333333", linewidth=1)

    axes[2].set_title(
        "Variables que influyeron en la predicción",
        fontsize=13,
        weight="bold"
    )

    axes[2].set_xlabel("Valor SHAP local")
    axes[2].set_ylabel("")

    for index, value in enumerate(contribuciones["Valor SHAP"]):
        desplazamiento = 0.02 if value > 0 else -0.02
        alineacion = "left" if value > 0 else "right"

        axes[2].text(
            value + desplazamiento,
            index,
            f"{value:.3f}",
            va="center",
            ha=alineacion,
            fontsize=9
        )

    plt.tight_layout()

    nombre_figura = f"salida_interpretativa_sistema_{deportista_id.replace(' ', '_')}.png"

    plt.savefig(
        nombre_figura,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return contribuciones

In [ ]:
import matplotlib.patches as patches

def dibujar_deportista_generico(ax):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    color_principal = "#4B6F8C"
    color_secundario = "#C7A76C"
    color_suave = "#E8ECEF"

    # Fondo circular
    circulo_fondo = patches.Circle(
        (0.5, 0.55),
        0.38,
        facecolor=color_suave,
        edgecolor="none"
    )
    ax.add_patch(circulo_fondo)

    # Cabeza
    cabeza = patches.Circle(
        (0.50, 0.78),
        0.07,
        facecolor=color_principal,
        edgecolor="none"
    )
    ax.add_patch(cabeza)

    # Tronco
    ax.plot(
        [0.50, 0.46],
        [0.70, 0.50],
        color=color_principal,
        linewidth=5,
        solid_capstyle="round"
    )

    # Brazo delantero
    ax.plot(
        [0.48, 0.66],
        [0.65, 0.58],
        color=color_principal,
        linewidth=4,
        solid_capstyle="round"
    )

    # Brazo posterior
    ax.plot(
        [0.48, 0.34],
        [0.64, 0.56],
        color=color_principal,
        linewidth=4,
        solid_capstyle="round"
    )

    # Pierna delantera
    ax.plot(
        [0.46, 0.66],
        [0.50, 0.32],
        color=color_principal,
        linewidth=5,
        solid_capstyle="round"
    )

    # Pierna posterior
    ax.plot(
        [0.46, 0.32],
        [0.50, 0.30],
        color=color_principal,
        linewidth=5,
        solid_capstyle="round"
    )

    # Línea de suelo
    ax.plot(
        [0.22, 0.78],
        [0.24, 0.24],
        color="#B8C0C6",
        linewidth=2
    )

    # Pequeño marcador deportivo
    marcador = patches.Circle(
        (0.78, 0.29),
        0.045,
        facecolor=color_secundario,
        edgecolor="white",
        linewidth=1
    )
    ax.add_patch(marcador)

    ax.text(
        0.5,
        0.08,
        "Deportista universitario",
        ha="center",
        va="center",
        fontsize=10,
        weight="bold",
        color="#2F3A40"
    )

In [ ]:
def generar_salida_sistema_con_dibujo(indice_caso=0):
    caso_original = X_test.iloc[[indice_caso]].copy()
    y_real_caso = int(y_test.iloc[indice_caso])

    deportista_id = f"Caso {indice_caso + 1:02d}"

    prob_alto = pipeline_logistica.predict_proba(caso_original)[0, 1]
    pred_binaria = int(prob_alto >= 0.50)

    clase_real = "Alto rendimiento" if y_real_caso == 1 else "En desarrollo"
    clase_predicha = "Alto rendimiento" if pred_binaria == 1 else "En desarrollo"

    caso_proc = preprocesador_logistica.transform(caso_original)
    shap_caso = explainer.shap_values(caso_proc)[0]

    contribuciones = pd.DataFrame({
        "Variable": nombres_variables_proc,
        "Valor SHAP": shap_caso
    })

    contribuciones["Importancia absoluta"] = contribuciones["Valor SHAP"].abs()
    contribuciones["Variable legible"] = contribuciones["Variable"].apply(nombre_legible_variable)

    contribuciones = (
        contribuciones
        .sort_values("Importancia absoluta", ascending=False)
        .head(8)
        .sort_values("Valor SHAP", ascending=True)
    )

    colores = contribuciones["Valor SHAP"].apply(
        lambda x: "#4B6F8C" if x > 0 else "#C7A76C"
    )

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5.5),
        dpi=150,
        gridspec_kw={"width_ratios": [0.9, 1.0, 1.5]}
    )

    # Panel 1: dibujo
    dibujar_deportista_generico(axes[0])
    axes[0].set_title("Deportista evaluado", fontsize=13, weight="bold", pad=10)
    axes[0].text(
        0.5, -0.05,
        deportista_id,
        ha="center",
        transform=axes[0].transAxes,
        fontsize=12,
        weight="bold"
    )

    # Panel 2: resultado
    axes[1].axis("off")

    axes[1].text(0.02, 0.92, "Resultado del sistema", fontsize=14, weight="bold")
    axes[1].text(0.02, 0.74, f"Clase real:\n{clase_real}", fontsize=11)

    axes[1].text(
        0.02, 0.56,
        f"Clase predicha:\n{clase_predicha}",
        fontsize=11,
        weight="bold",
        color="#2F4F5F"
    )

    axes[1].text(
        0.02, 0.38,
        f"Probabilidad de alto rendimiento:\n{prob_alto:.1%}",
        fontsize=11
    )

    axes[1].barh([0], [prob_alto], color="#4B6F8C", height=0.18)
    axes[1].barh([0], [1 - prob_alto], left=[prob_alto], color="#D9DEE2", height=0.18)

    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(-0.5, 1)

    # Panel 3: SHAP local
    axes[2].barh(
        contribuciones["Variable legible"],
        contribuciones["Valor SHAP"],
        color=colores
    )

    axes[2].axvline(0, color="#333333", linewidth=1)
    axes[2].set_title("Variables que influyeron en la predicción", fontsize=13, weight="bold")
    axes[2].set_xlabel("Valor SHAP local")
    axes[2].set_ylabel("")

    plt.tight_layout()

    plt.savefig(
        f"salida_interpretativa_sistema_{deportista_id.replace(' ', '_')}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return contribuciones

In [ ]:
contribuciones_caso = generar_salida_sistema_con_dibujo(indice_caso=0)
contribuciones_caso

In [ ]:
import os

os.makedirs("resultados_morphofusion", exist_ok=True)
os.makedirs("resultados_morphofusion/figuras", exist_ok=True)
os.makedirs("resultados_morphofusion/tablas", exist_ok=True)

In [ ]:
tabla_comparacion_final.to_excel(
    "resultados_morphofusion/tablas/tabla_comparacion_final_7_modelos.xlsx",
    index=False
)

tabla_metricas_dl.to_excel(
    "resultados_morphofusion/tablas/tabla_metricas_modelos_dl.xlsx",
    index=False
)

tabla_metricas_ml_final.to_excel(
    "resultados_morphofusion/tablas/tabla_metricas_modelos_ml.xlsx",
    index=False
)

importancia_shap.to_excel(
    "resultados_morphofusion/tablas/tabla_importancia_shap.xlsx",
    index=False
)

importancia_logistica_dim.to_excel(
    "resultados_morphofusion/tablas/tabla_coeficientes_regresion_logistica.xlsx",
    index=False
)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "resultados_morphofusion",
    "zip",
    "resultados_morphofusion"
)

files.download("resultados_morphofusion.zip")

In [ ]:
# Guardar matriz completa de Pearson en Excel
matriz_pearson.to_excel(
    "matriz_correlacion_pearson_completa.xlsx"
)

files.download("matriz_correlacion_pearson_completa.xlsx")

In [ ]:
pares_correlacionados.to_excel(
    "tabla_pares_alta_correlacion_pearson.xlsx",
    index=False
)

files.download("tabla_pares_alta_correlacion_pearson.xlsx")

In [ ]:
files.download("figura_matriz_pearson_reducida_revista.png")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

fig, axes = plt.subplots(2, 2, figsize=(9, 8), dpi=180)
axes = axes.flatten()

orden_modelos_ml = [
    "Regresión Logística",
    "SVM",
    "Random Forest",
    "XGBoost"
]

for ax, modelo in zip(axes, orden_modelos_ml):

    matriz = matrices_ml_final[modelo]

    sns.heatmap(
        matriz,
        annot=True,
        fmt="d",
        cmap=sns.light_palette("#4B6F8C", as_cmap=True),
        cbar=False,
        xticklabels=["En desarrollo", "Alto rendimiento"],
        yticklabels=["En desarrollo", "Alto rendimiento"],
        annot_kws={"size": 16, "weight": "bold"},
        linewidths=0.8,
        linecolor="white",
        ax=ax
    )

    ax.set_title(modelo, fontsize=12, weight="bold", pad=10)
    ax.set_xlabel("Predicción", fontsize=10)
    ax.set_ylabel("Valor real", fontsize=10)
    ax.tick_params(axis="x", labelsize=9, rotation=0)
    ax.tick_params(axis="y", labelsize=9, rotation=90)

plt.suptitle(
    "Matrices de confusión de los modelos de Machine Learning",
    fontsize=14,
    weight="bold",
    y=1.03
)

plt.tight_layout(pad=2.0)

nombre_archivo = "figura_matrices_confusion_modelos_ml_final.png"

plt.savefig(
    nombre_archivo,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

files.download(nombre_archivo)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

metricas_cv = {
    "Accuracy": "accuracy",
    "Precisión": "precision",
    "Recall": "recall",
    "F1-score": "f1",
    "AUC-ROC": "roc_auc"
}

resultados_cv_formato = []
resultados_cv_simple = []

for nombre_modelo, modelo in modelos_ml.items():

    pipeline_cv = Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", modelo)
    ])

    scores = cross_validate(
        pipeline_cv,
        X,
        y,
        cv=cv5,
        scoring=metricas_cv,
        return_train_score=False
    )

    # Tabla numérica para gráficos
    resultados_cv_simple.append({
        "Modelo": nombre_modelo,
        "Accuracy": scores["test_Accuracy"].mean(),
        "Precisión": scores["test_Precisión"].mean(),
        "Recall": scores["test_Recall"].mean(),
        "F1-score": scores["test_F1-score"].mean(),
        "AUC-ROC": scores["test_AUC-ROC"].mean()
    })

    # Tabla con media ± desviación estándar para artículo
    resultados_cv_formato.append({
        "Modelo": nombre_modelo,
        "Accuracy": f"{scores['test_Accuracy'].mean():.3f} ± {scores['test_Accuracy'].std():.3f}",
        "Precisión": f"{scores['test_Precisión'].mean():.3f} ± {scores['test_Precisión'].std():.3f}",
        "Recall": f"{scores['test_Recall'].mean():.3f} ± {scores['test_Recall'].std():.3f}",
        "F1-score": f"{scores['test_F1-score'].mean():.3f} ± {scores['test_F1-score'].std():.3f}",
        "AUC-ROC": f"{scores['test_AUC-ROC'].mean():.3f} ± {scores['test_AUC-ROC'].std():.3f}"
    })

resultados_cv_simple = pd.DataFrame(resultados_cv_simple)
resultados_cv_formato = pd.DataFrame(resultados_cv_formato)

resultados_cv_simple = resultados_cv_simple.sort_values(
    "F1-score",
    ascending=False
)

resultados_cv_formato = resultados_cv_formato.set_index("Modelo").loc[
    resultados_cv_simple["Modelo"]
].reset_index()

resultados_cv_formato

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

tabla_cv_plot = resultados_cv_simple.copy()
tabla_cv_plot = tabla_cv_plot.sort_values("F1-score", ascending=True)

plt.figure(figsize=(8.5, 4.8), dpi=150)

sns.barplot(
    data=tabla_cv_plot,
    x="F1-score",
    y="Modelo",
    color="#5E6A71"
)

plt.title(
    "Validación cruzada estratificada de 5 pliegues",
    fontsize=13,
    weight="bold"
)

plt.xlabel("F1-score promedio")
plt.ylabel("")
plt.xlim(0, 1)

for index, value in enumerate(tabla_cv_plot["F1-score"]):
    plt.text(
        value + 0.01,
        index,
        f"{value:.3f}",
        va="center",
        fontsize=10
    )

sns.despine(left=True, bottom=False)
plt.tight_layout()

nombre_figura_cv = "figura_validacion_cruzada_5fold_f1_ml.png"

plt.savefig(
    nombre_figura_cv,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

files.download(nombre_figura_cv)

In [ ]:
from google.colab import files

# Descargar figura de dirección de influencia
files.download("figura_direccion_influencia_regresion_logistica_revista.png")

# Descargar SHAP Summary Plot
files.download("figura_shap_summary_plot_nombres_reales.png")

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import pandas as pd

def generar_salida_sistema_con_dibujo(indice_caso=0):
    caso_original = X_test.iloc[[indice_caso]].copy()
    y_real_caso = int(y_test.iloc[indice_caso])

    deportista_id = f"Caso {indice_caso + 1:02d}"

    prob_alto = pipeline_logistica.predict_proba(caso_original)[0, 1]
    pred_binaria = int(prob_alto >= 0.50)

    clase_real = "Alto rendimiento" if y_real_caso == 1 else "En desarrollo"
    clase_predicha = "Alto rendimiento" if pred_binaria == 1 else "En desarrollo"

    caso_proc = preprocesador_logistica.transform(caso_original)
    shap_caso = explainer.shap_values(caso_proc)[0]

    contribuciones = pd.DataFrame({
        "Variable": nombres_variables_proc,
        "Valor SHAP": shap_caso
    })

    contribuciones["Importancia absoluta"] = contribuciones["Valor SHAP"].abs()
    contribuciones["Variable legible"] = contribuciones["Variable"].apply(nombre_legible_variable)

    contribuciones = (
        contribuciones
        .sort_values("Importancia absoluta", ascending=False)
        .head(8)
        .sort_values("Valor SHAP", ascending=True)
    )

    colores = contribuciones["Valor SHAP"].apply(
        lambda x: "#4B6F8C" if x > 0 else "#C7A76C"
    )

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5.5),
        dpi=150,
        gridspec_kw={"width_ratios": [0.9, 1.0, 1.5]}
    )

    # Panel 1: dibujo
    dibujar_deportista_generico(axes[0])
    axes[0].set_title("Deportista evaluado", fontsize=13, weight="bold", pad=10)
    axes[0].text(
        0.5, -0.05,
        deportista_id,
        ha="center",
        transform=axes[0].transAxes,
        fontsize=12,
        weight="bold"
    )

    # Panel 2: resultado
    axes[1].axis("off")

    axes[1].text(0.02, 0.92, "Resultado del sistema", fontsize=14, weight="bold")
    axes[1].text(0.02, 0.74, f"Clase real:\n{clase_real}", fontsize=11)

    axes[1].text(
        0.02, 0.56,
        f"Clase predicha:\n{clase_predicha}",
        fontsize=11,
        weight="bold",
        color="#2F4F5F"
    )

    axes[1].text(
        0.02, 0.38,
        f"Probabilidad de alto rendimiento:\n{prob_alto:.1%}",
        fontsize=11
    )

    axes[1].barh([0], [prob_alto], color="#4B6F8C", height=0.18)
    axes[1].barh([0], [1 - prob_alto], left=[prob_alto], color="#D9DEE2", height=0.18)

    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(-0.5, 1)

    # Panel 3: SHAP local
    axes[2].barh(
        contribuciones["Variable legible"],
        contribuciones["Valor SHAP"],
        color=colores
    )

    axes[2].axvline(0, color="#333333", linewidth=1)
    axes[2].set_title(
        "Variables que influyeron en la predicción",
        fontsize=13,
        weight="bold"
    )
    axes[2].set_xlabel("Valor SHAP local")
    axes[2].set_ylabel("")

    plt.tight_layout()

    nombre_archivo = f"salida_interpretativa_sistema_{deportista_id.replace(' ', '_')}.png"

    fig.savefig(
        nombre_archivo,
        dpi=300,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.show()

    files.download(nombre_archivo)

    return contribuciones

In [ ]:
contribuciones_caso = generar_salida_sistema_con_dibujo(indice_caso=0)